# Flood Detection — Prithvi-EO-2.0-300M-TL (Sen1Floods11 Finetuned Transfer)
### AISEHack Theme 1 | Transfer Learning from Flood-Finetuned Foundation Model

This notebook uses the **full finetuned weights** from [Prithvi-EO-2.0-300M-TL-Sen1Floods11](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11) — a ViT-Large encoder + UperNet decoder already trained for flood segmentation.

**Key differences from the 600M backbone-only notebook:**
- Loads BOTH the encoder AND the UperNet decoder pretrained for flood segmentation
- The decoder has already learned to segment water/flood from ViT features
- Smart patch embed transfer copies pretrained filters for matching optical bands
- Smaller model: ViT-Large (1024-dim, 24 layers) vs ViT-Huge (1280-dim, 32 layers)

**Model Architecture (from HuggingFace config):**
- Backbone: `prithvi_eo_v2_300_tl` — ViT-Large (1024-dim, 24 layers, patch_size=16)
- Decoder: UperNet (256 channels, scale_modules=True)
- Feature indices: layers [5, 11, 17, 23]
- Pretrained bands: [BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2]
- Weights file: `Prithvi-EO-V2-300M-TL-Sen1Floods11.pt` (1.28 GB)

**Our dataset bands:** [HH, HV, Green, Red, NIR, SWIR]
- 4 of 6 pretrained bands match: Green→GREEN, Red→RED, NIR→NIR_NARROW, SWIR→SWIR_1
- 2 bands are new (SAR): HH, HV → randomly initialized in patch embedding

In [ ]:

# ── Install dependencies (Kaggle) ──
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightning',
    'einops',
    'huggingface_hub',
    'rasterio',
    'scipy',
])

# ── Conditional TPU support ──
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    _XLA_AVAILABLE = True
except ImportError:
    _XLA_AVAILABLE = False

import timm, torch, lightning
print(f'torch      {torch.__version__}')
print(f'timm       {timm.__version__}')
print(f'lightning   {lightning.__version__}')
print(f'numpy      {__import__("numpy").__version__}')
print(f'torch_xla  {"✓ " + torch_xla.__version__ if _XLA_AVAILABLE else "✗ not installed"}')
print(f'\n✓ Ready — no terratorch needed.')


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import os, sys, json, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import rasterio
import timm
from scipy.ndimage import convolve as _sci_convolve

import albumentations as A
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy

import matplotlib.pyplot as plt
import pandas as pd

USE_TPU    = False            # ← Toggle TPU on/off here
TPU_CORES  = 8

# ── Device helper (TPU / CUDA / CPU) ──
def get_device():
    if USE_TPU and _XLA_AVAILABLE:
        return xm.xla_device()
    elif torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def empty_accelerator_cache():
    if USE_TPU and _XLA_AVAILABLE:
        xm.mark_step()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
if USE_TPU:
    if _XLA_AVAILABLE:
        print(f'TPU: enabled ({TPU_CORES} core(s))')
    else:
        print('⚠ USE_TPU=True but torch_xla not installed — falling back to GPU/CPU')


In [ ]:

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║        PRITHVI-EO-2.0-300M-TL-SEN1FLOODS11 CONFIG                      ║
# ║                                                                        ║
# ║  Full finetuned transfer: encoder + decoder pretrained on Sen1Floods11. ║
# ║  Decoder + head already know flood segmentation → lower decoder LR.    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Model Source ──
HF_REPO      = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL-Sen1Floods11'
HF_CKPT_FILE = 'Prithvi-EO-V2-300M-TL-Sen1Floods11.pt'

# ── Architecture (from config.json) ──
BACKBONE       = 'prithvi_eo_v2_300_tl'
EMBED_DIM      = 1024           # ViT-Large
DEPTH          = 24
OUT_INDICES    = (5, 11, 17, 23) # Feature extraction layers (evenly spaced)
PATCH_SIZE     = 16
TIMM_MODEL     = 'vit_large_patch16_224'
DECODER        = 'upernet'
DECODER_CH     = 256
NUM_CLASSES    = 2

# ── Hyperparameters ──
EPOCHS         = 60
BATCH_SIZE     = 4              # larger batch — 300M is ~0.5× the 600M model
ACCUM_GRAD     = 4              # effective batch = 16
LR             = 3e-5
WEIGHT_DECAY   = 0.05
HEAD_DROPOUT   = 0.1
CLASS_WEIGHTS  = [0.25, 0.75]  # flood pixels ~3–9× rarer → stronger boost
SEED           = 42
NUM_WORKERS    = 4
IMG_SIZE       = 224            # match pretrained img_size
WARMUP_ITERS   = 15
WARMUP_RATIO   = 1e-6
ES_PATIENCE    = 35
FREEZE_BACKBONE = False
AUX_LOSS_WEIGHT = 0.4           # auxiliary loss weight (standard MMSeg/PSPNet)

# ── Transfer mode ──
# 'full'   : load both encoder + decoder weights, finetune everything
# 'encoder': load only encoder weights, decoder random init
# 'frozen' : load both, freeze encoder, only train decoder
TRANSFER_MODE = 'full'

# ── Pretrained Sen1Floods11 weight loading ──
LOAD_SEN1FLOODS11 = False   # Set False to skip pretrained weights (random init)

# ── Channels: raw 6 bands, SAR in dB ──
RAW_BANDS = ['HH', 'HV', 'Green', 'Red', 'NIR', 'SWIR']

# ── SAR preprocessing toggle ──
# True  = Refined Lee speckle filter → dB  (edge-preserving denoise, then log-scale)
# False = dB only                          (let the model learn to handle speckle)
USE_REFINED_LEE = False

# ── Experiment Config ──
_sar_tag = 'sardb' if USE_REFINED_LEE else 'sardb_nolee'
EXP_CFG = {
    'name': f'v2_300m_tl_6ch_{_sar_tag}',
    'desc': f'Prithvi-EO-2.0-300M-TL-Sen1Floods11 + UperNet, 6 bands (SAR {"Refined Lee + dB" if USE_REFINED_LEE else "dB only"})',
    'raw_bands': RAW_BANDS,
    'indices': [],
    'use_dem': False,
}


ch_names = EXP_CFG['raw_bands']

N_CH = len(ch_names)

print(f'Model: {HF_REPO}')
print(f'Architecture: {TIMM_MODEL} (embed_dim={EMBED_DIM}, depth={DEPTH}, patch={PATCH_SIZE})')
print(f'LR: {LR} | Epochs: {EPOCHS} | BS: {BATCH_SIZE}x{ACCUM_GRAD}={BATCH_SIZE*ACCUM_GRAD}')
print(f'Channels ({N_CH}): {ch_names}')
print(f'SAR preprocessing: {"Refined Lee + dB" if USE_REFINED_LEE else "dB only (no filter)"}')
print(f'Transfer mode: {TRANSFER_MODE} | Load Sen1Floods11: {LOAD_SEN1FLOODS11}')

In [ ]:
# ── Path Configuration (auto-detect Kaggle vs local) ──
ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    DATA_ROOT    = Path('/kaggle/input/competitions/aisehack-theme-1/data')
    DEM_ROOT     = Path('/kaggle/input/datasets/yashsorathiya009/flood-dem-data/dem')
    OUTPUT_DIR   = Path('/kaggle/working/experiments')
else:
    DATA_ROOT    = Path('C:/Users/sorat/Downloads/aisehack-theme-1/data')
    DEM_ROOT     = DATA_ROOT / 'dem'
    OUTPUT_DIR   = Path('C:/Users/sorat/Downloads/aisehack-theme-1/outputs')

IMG_DIR      = DATA_ROOT / 'image'
LBL_DIR      = DATA_ROOT / 'label'
PRED_IMG_DIR = DATA_ROOT / 'prediction' / 'image'
SPLIT_DIR    = DATA_ROOT / 'split'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load split files ──
def load_split(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_files = load_split(SPLIT_DIR / 'train.txt')
val_files   = load_split(SPLIT_DIR / 'val.txt')
test_files  = load_split(SPLIT_DIR / 'test.txt')
pred_files  = load_split(SPLIT_DIR / 'pred.txt')

print(f'Splits  — train: {len(train_files)}, val: {len(val_files)}, test: {len(test_files)}, pred: {len(pred_files)}')
print(f'Data    — {DATA_ROOT}')
print(f'Output  — {OUTPUT_DIR}')

In [ ]:

# ══════════════════════════════════════════════════════════════
#  Channel Engineering — Refined Lee filter + dB for SAR
# ══════════════════════════════════════════════════════════════

RAW_BAND_IDX = {'HH': 0, 'HV': 1, 'Green': 2, 'Red': 3, 'NIR': 4, 'SWIR': 5}
SAR_BANDS = {'HH', 'HV'}  # bands that get speckle filter + dB transform


# ── Refined Lee Speckle Filter ──────────────────────────────

def _build_direction_kernels(size=7):
    """Build 8 directional line kernels for the Refined Lee filter.
    Each kernel selects pixels along one edge direction within a size×size window.
    Directions at 0°, 22.5°, 45°, … , 157.5° (perpendicular pairs share stats)."""
    half = size // 2
    kernels = []
    for angle_deg in [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5]:
        k = np.zeros((size, size), dtype=np.float64)
        angle_rad = np.deg2rad(angle_deg)
        # Normal to the line direction (used to compute distance to line)
        nx, ny = np.cos(angle_rad), np.sin(angle_rad)
        for i in range(size):
            for j in range(size):
                di, dj = i - half, j - half
                dist = abs(di * nx + dj * ny)
                if dist <= 1.0:
                    k[i, j] = 1.0
        k[half, half] = 1.0  # always include center
        kernels.append(k)
    return kernels

# Pre-compute kernels once (7×7 window)
_LEE_KERNELS_7 = _build_direction_kernels(7)


def refined_lee_filter(band, window_size=7, n_looks=1):
    """Refined Lee speckle filter for a single SAR intensity band.

    1. Compute local mean & variance in 8 directional sub-windows.
    2. Select the direction with lowest coefficient of variation (most
       homogeneous = best edge alignment) per pixel.
    3. Apply MMSE filter:  result = mean + w * (pixel - mean)
       where w = clamp((local_var - noise_var) / local_var, 0, 1)
       and noise_var = mean² / n_looks (multiplicative speckle model).

    Parameters
    ----------
    band : 2-D ndarray   – SAR intensity (linear power, NOT dB)
    window_size : int     – filter kernel size (default 7)
    n_looks : float       – equivalent number of looks (default 1)
    """
    if window_size == 7:
        kernels = _LEE_KERNELS_7
    else:
        kernels = _build_direction_kernels(window_size)

    b = band.astype(np.float64)
    b_sq = b ** 2

    best_cv  = np.full(b.shape, np.inf, dtype=np.float64)
    best_mean = np.zeros_like(b)
    best_var  = np.zeros_like(b)

    for k in kernels:
        k_norm = k / k.sum()
        lm  = _sci_convolve(b,    k_norm, mode='reflect')
        lsq = _sci_convolve(b_sq, k_norm, mode='reflect')
        lv  = np.maximum(lsq - lm ** 2, 0.0)
        cv  = np.sqrt(lv) / (np.abs(lm) + 1e-10)

        better = cv < best_cv
        best_cv[better]   = cv[better]
        best_mean[better] = lm[better]
        best_var[better]  = lv[better]

    # Noise variance under multiplicative speckle model
    noise_var = (best_mean ** 2) / max(n_looks, 1.0)

    # MMSE weight (0 → use local mean, 1 → keep original pixel)
    weight = np.clip((best_var - noise_var) / (best_var + 1e-10), 0.0, 1.0)

    result = best_mean + weight * (b - best_mean)
    return result.astype(np.float32)


# ── dB transform ────────────────────────────────────────────

def sar_to_db(x):
    """Convert SAR backscatter (linear power) to decibels.
    Clips at 1e-10 to avoid log(0)."""
    return (10.0 * np.log10(np.clip(x, 1e-10, None))).astype(np.float32)


# ── Channel stack builder ──────────────────────────────────

def build_channel_stack(img6, dem, exp_cfg):
    '''Assemble the channel tensor for a given experiment config.
    SAR bands: optionally Refined Lee filter → dB scale, then stack.'''
    channels = []
    for b in exp_cfg['raw_bands']:
        band = img6[RAW_BAND_IDX[b]]
        if b in SAR_BANDS:
            if USE_REFINED_LEE:
                band = refined_lee_filter(band)  # speckle suppression (linear domain)
            band = sar_to_db(band)               # convert to dB
        channels.append(band)
    return np.stack(channels, axis=0).astype(np.float32)


# ── Preprocessed channel cache ─────────────────────────────
# Saves build_channel_stack output (Refined Lee + dB) as .npy
# so the expensive SAR filtering runs once, not every epoch.

def _get_cache_dir():
    '''Return cache directory path (inside OUTPUT_DIR to avoid polluting data/).'''
    d = OUTPUT_DIR / 'channel_cache' / EXP_CFG['name']
    d.mkdir(parents=True, exist_ok=True)
    return d


def precompute_channel_cache(file_list, img_dir, exp_cfg, label=''):
    '''Pre-compute and cache the channel stack for every image.
    Skips files that are already cached.  Returns the cache dir.'''
    cache_dir = _get_cache_dir()
    todo = [f for f in file_list if not (cache_dir / f'{f}.npy').exists()]
    if not todo:
        print(f'  Channel cache ({label}): all {len(file_list)} files already cached.')
        return cache_dir
    print(f'  Channel cache ({label}): processing {len(todo)}/{len(file_list)} files...')
    t0 = time.time()
    for i, fname in enumerate(todo):
        with rasterio.open(img_dir / f'{fname}_image.tif') as src:
            img6 = src.read().astype(np.float32)
        ch = build_channel_stack(img6, None, exp_cfg)
        np.save(cache_dir / f'{fname}.npy', ch)
        if (i + 1) % 20 == 0:
            print(f'    {i+1}/{len(todo)} done...')
    dt = time.time() - t0
    print(f'  Channel cache ({label}): {len(todo)} files cached in {dt:.1f}s')
    return cache_dir


def load_cached_channels(fname, cache_dir):
    '''Load pre-computed channel stack from cache.  Returns None on miss.'''
    npy_path = cache_dir / f'{fname}.npy'
    if npy_path.exists():
        return np.load(npy_path)
    return None


def compute_norm_stats(exp_cfg, file_list, img_dir):
    '''Compute per-channel mean & std over training set (uses cache).'''
    cache_dir = _get_cache_dir()
    sums, sq_sums, n_pix = None, None, 0
    for fname in file_list:
        ch = load_cached_channels(fname, cache_dir)
        if ch is None:
            with rasterio.open(img_dir / f'{fname}_image.tif') as src:
                img6 = src.read().astype(np.float32)
            ch = build_channel_stack(img6, None, exp_cfg)
        npx = ch.shape[1] * ch.shape[2]
        if sums is None:
            sums    = np.zeros(ch.shape[0], dtype=np.float64)
            sq_sums = np.zeros(ch.shape[0], dtype=np.float64)
        for c in range(ch.shape[0]):
            sums[c]    += ch[c].sum()
            sq_sums[c] += (ch[c].astype(np.float64) ** 2).sum()
        n_pix += npx
    means = sums / n_pix
    stds  = np.sqrt(sq_sums / n_pix - means ** 2)
    return means.tolist(), stds.tolist()


# ── Quick test ──
with rasterio.open(IMG_DIR / f'{train_files[0]}_image.tif') as src:
    _img = src.read().astype(np.float32)

t0 = time.time()
_ch = build_channel_stack(_img, None, EXP_CFG)
dt = time.time() - t0

print(f'Channel stack test — shape: {_ch.shape}, bands: {EXP_CFG["raw_bands"]}')
print(f'  Pipeline: Raw SAR → Refined Lee (7×7) → dB → stack')
print(f'  HH dB range: [{_ch[0].min():.2f}, {_ch[0].max():.2f}]')
print(f'  HV dB range: [{_ch[1].min():.2f}, {_ch[1].max():.2f}]')
print(f'  Processing time per image: {dt:.2f}s')

In [ ]:

# ══════════════════════════════════════════════════════════════
#  Dataset & DataModule
# ══════════════════════════════════════════════════════════════

class FloodDataset(Dataset):
    '''Flood segmentation dataset with configurable channels.'''

    def __init__(self, file_list, img_dir, lbl_dir,
                 exp_cfg, means, stds, transform=None, is_predict=False,
                 cache_dir=None):
        self.file_list  = file_list
        self.img_dir    = Path(img_dir)
        self.lbl_dir    = Path(lbl_dir) if lbl_dir else None
        self.exp_cfg    = exp_cfg
        self.means      = np.array(means, dtype=np.float32).reshape(-1, 1, 1)
        self.stds       = np.array(stds,  dtype=np.float32).reshape(-1, 1, 1)
        self.transform  = transform
        self.is_predict = is_predict
        self.cache_dir  = Path(cache_dir) if cache_dir else None

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]

        # Try loading from cache first (skips Refined Lee + dB)
        channels = None
        if self.cache_dir is not None:
            channels = load_cached_channels(fname, self.cache_dir)
        if channels is None:
            with rasterio.open(self.img_dir / f'{fname}_image.tif') as src:
                img6 = src.read().astype(np.float32)
            channels = build_channel_stack(img6, None, self.exp_cfg)
        channels = (channels - self.means) / (self.stds + 1e-8)

        mask = None
        if not self.is_predict and self.lbl_dir:
            lp = self.lbl_dir / f'{fname}_label.tif'
            if lp.exists():
                with rasterio.open(lp) as src:
                    mask = src.read(1).astype(np.int64)

        if self.transform:
            ch_hwc = channels.transpose(1, 2, 0)
            if mask is not None:
                aug = self.transform(image=ch_hwc, mask=mask)
                ch_hwc, mask = aug['image'], aug['mask']
            else:
                ch_hwc = self.transform(image=ch_hwc)['image']
            if isinstance(ch_hwc, torch.Tensor):
                channels = ch_hwc
            else:
                channels = torch.from_numpy(ch_hwc.transpose(2, 0, 1).copy())
        else:
            channels = torch.from_numpy(channels.copy())

        channels = channels.float()
        out = {'image': channels, 'filename': fname}
        if mask is not None:
            out['mask'] = torch.from_numpy(mask.copy()).long() if isinstance(mask, np.ndarray) else mask.long()
        else:
            out['mask'] = torch.full((channels.shape[1], channels.shape[2]), -1, dtype=torch.long)
        return out


class FloodDataModule(pl.LightningDataModule):

    def __init__(self, exp_cfg, means, stds,
                 train_files, val_files, test_files, pred_files,
                 img_dir, lbl_dir, pred_img_dir,
                 batch_size=2, num_workers=2, img_size=224):
        super().__init__()
        self.exp_cfg      = exp_cfg
        self.means        = means
        self.stds         = stds
        self.train_files  = train_files
        self.val_files    = val_files
        self.test_files   = test_files
        self.pred_files   = pred_files
        self.img_dir      = img_dir
        self.lbl_dir      = lbl_dir
        self.pred_img_dir = pred_img_dir
        self.batch_size   = batch_size
        self.num_workers  = num_workers
        self.img_size     = img_size

        if img_size < 512:
            self.train_aug = A.Compose([
                A.RandomCrop(height=img_size, width=img_size),
                A.D4(),
                A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
                A.GaussianBlur(blur_limit=(3, 7), p=0.3),
                A.RandomBrightnessContrast(
                    brightness_limit=0.15, contrast_limit=0.15, p=0.3),
                A.CoarseDropout(
                    num_holes_range=(1, 3), hole_height_range=(16, 32),
                    hole_width_range=(16, 32), fill=0, p=0.2),
                ToTensorV2(),
            ])
            # No crop for eval — full 512×512 fed to sliding-window val (matches submission)
            self.eval_aug = A.Compose([ToTensorV2()])
        else:
            self.train_aug = A.Compose([A.D4(), ToTensorV2()])
            self.eval_aug = A.Compose([ToTensorV2()])

    def setup(self, stage=None):
        cache_dir = _get_cache_dir()
        if stage in ('fit', None):
            # Repeat train list 4× so each image yields 4 different RandomCrops per epoch
            train_files_4x = self.train_files * 8
            self.train_ds = FloodDataset(
                train_files_4x, self.img_dir, self.lbl_dir,
                self.exp_cfg, self.means, self.stds, self.train_aug,

                cache_dir=cache_dir)
            self.val_ds = FloodDataset(
                self.val_files, self.img_dir, self.lbl_dir,
                self.exp_cfg, self.means, self.stds, self.eval_aug,

                cache_dir=cache_dir)
        if stage in ('test', None):
            self.test_ds = FloodDataset(
                self.test_files, self.img_dir, self.lbl_dir,
                self.exp_cfg, self.means, self.stds, self.eval_aug,

                cache_dir=cache_dir)
        if stage in ('predict', None):
            self.pred_ds = FloodDataset(
                self.pred_files, self.pred_img_dir, None,
                self.exp_cfg, self.means, self.stds, self.eval_aug,
                is_predict=True, cache_dir=cache_dir)


    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size,
                          shuffle=True, num_workers=self.num_workers,
                          pin_memory=True, drop_last=True)
    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True)
    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True)
    def predict_dataloader(self):
        return DataLoader(self.pred_ds, batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True)

print(f'FloodDataset & FloodDataModule defined (img_size={IMG_SIZE}).')
print(f'  Train augmentation: RandomCrop({IMG_SIZE}) + D4 + noise/blur/brightness/dropout')
print(f'  Eval augmentation:  ToTensorV2() only — full image passed to sliding-window val')


In [ ]:

# ══════════════════════════════════════════════════════════════════════════
#  Model Architecture — ViT-Large Encoder + UperNet Decoder + SegHead
#
#  Prithvi-EO-2.0-300M-TL-Sen1Floods11 architecture:
#    - Backbone: ViT-Large (embed_dim=1024, depth=24, patch_size=16)
#    - Feature indices: [5, 11, 17, 23]
#    - Decoder: UperNet (channels=256, scale_modules=True)
#    - Head: SegmentationHead (dropout=0.1, num_classes=2)
# ══════════════════════════════════════════════════════════════════════════

from timm.models.vision_transformer import Block


# ── Smart Patch Embed Transfer ──
# Maps our band names → the pretrained model's band indices.
# Pretrained order: [BLUE(0), GREEN(1), RED(2), NIR_NARROW(3), SWIR_1(4), SWIR_2(5)]
BAND_TO_PRETRAINED_INDEX = {
    'Green': 1,   # Our Green  ← pretrained GREEN filter
    'Red':   2,   # Our Red    ← pretrained RED filter
    'NIR':   3,   # Our NIR    ← pretrained NIR_NARROW filter
    'SWIR':  4,   # Our SWIR   ← pretrained SWIR_1 filter
}


class ViTEncoder(nn.Module):
    """Wraps a timm ViT to extract multi-layer 2D feature maps.

    For Prithvi v2 300M: ViT-Large (1024-dim, 24 layers, patch_size=16).
    Extracts features at intermediate layers [5, 11, 17, 23] via hooks.
    """

    def __init__(self, timm_name, in_channels, img_size=224,
                 pretrained=False, out_indices=(5, 11, 17, 23)):
        super().__init__()
        self.vit = timm.create_model(
            timm_name, pretrained=pretrained,
            img_size=img_size, in_chans=in_channels, num_classes=0,
            dynamic_img_size=True,
        )
        self.embed_dim   = self.vit.embed_dim
        self.out_indices = out_indices
        pe = self.vit.patch_embed
        self.patch_size = (pe.patch_size[0]
                           if isinstance(pe.patch_size, (tuple, list))
                           else pe.patch_size)

        self._cache = {}
        for idx in out_indices:
            self.vit.blocks[idx].register_forward_hook(self._hook(idx))

    def _hook(self, idx):
        def fn(module, inp, out):
            self._cache[idx] = out
        return fn

    def forward(self, x):
        B, C, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size

        self.vit.forward_features(x)

        feats = []
        for idx in self.out_indices:
            tok = self._cache[idx]
            if tok.shape[1] != h * w:
                tok = tok[:, -h * w:]
            feats.append(tok.transpose(1, 2).reshape(B, self.embed_dim, h, w))
        return feats


# ══════════════════════════════════════════════════════════════════════════
#  UperNet Decoder
# ══════════════════════════════════════════════════════════════════════════

class ConvModule(nn.Module):
    """Conv2d + BatchNorm2d + ReLU."""
    def __init__(self, in_channels, out_channels, kernel_size=3,
                 padding=0, dilation=1, stride=1, inplace=False):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              padding=padding, dilation=dilation,
                              stride=stride, bias=False)
        self.norm = nn.BatchNorm2d(out_channels)
        self.act  = nn.ReLU(inplace=inplace)

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))


class PPM(nn.ModuleList):
    """Pooling Pyramid Module."""
    def __init__(self, pool_scales, in_channels, channels, align_corners=True):
        super().__init__()
        self.align_corners = align_corners
        for pool_scale in pool_scales:
            self.append(
                nn.Sequential(
                    nn.AdaptiveAvgPool2d(pool_scale),
                    ConvModule(in_channels, channels, 1, inplace=True),
                )
            )

    def forward(self, x):
        ppm_outs = []
        for ppm in self:
            ppm_out = ppm(x)
            upsampled = F.interpolate(
                ppm_out, size=x.size()[2:],
                mode='bilinear', align_corners=self.align_corners)
            ppm_outs.append(upsampled)
        return ppm_outs


class UperNetDecoder(nn.Module):
    """UperNet decoder with scale_modules for ViT backbones."""

    def __init__(self, embed_dim, pool_scales=(1, 2, 3, 6),
                 channels=256, align_corners=True, scale_modules=False):
        super().__init__()
        self.align_corners = align_corners
        self.out_channels  = channels
        self.channels      = channels

        if isinstance(embed_dim, int):
            embed_dim = [embed_dim] * 4

        # ── Scale Modules: ConvTranspose2d-based multi-scale FPN ──
        self._use_scale_modules = scale_modules
        if scale_modules:
            self.fpn1 = nn.Sequential(
                nn.ConvTranspose2d(embed_dim[0], embed_dim[0] // 2, 2, 2),
                nn.BatchNorm2d(embed_dim[0] // 2),
                nn.GELU(),
                nn.ConvTranspose2d(embed_dim[0] // 2, embed_dim[0] // 4, 2, 2))
            self.fpn2 = nn.Sequential(
                nn.ConvTranspose2d(embed_dim[1], embed_dim[1] // 2, 2, 2))
            self.fpn3 = nn.Sequential(nn.Identity())
            self.fpn4 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2))
            self.embed_dim = [embed_dim[0] // 4, embed_dim[1] // 2,
                              embed_dim[2], embed_dim[3]]
        else:
            self.embed_dim = embed_dim

        # ── PSP Module ──
        self.psp_modules = PPM(
            pool_scales, self.embed_dim[-1], self.channels,
            align_corners=self.align_corners)
        self.bottleneck = ConvModule(
            self.embed_dim[-1] + len(pool_scales) * self.channels,
            self.channels, 3, padding=1, inplace=True)

        # ── FPN: lateral (1×1) + smooth (3×3) convolutions ──
        self.lateral_convs = nn.ModuleList()
        self.fpn_convs     = nn.ModuleList()
        for ed in self.embed_dim[:-1]:
            self.lateral_convs.append(
                ConvModule(ed, self.channels, 1, inplace=False))
            self.fpn_convs.append(
                ConvModule(self.channels, self.channels, 3, padding=1, inplace=False))

        # ── FPN Bottleneck ──
        self.fpn_bottleneck = ConvModule(
            len(self.embed_dim) * self.channels, self.channels, 3,
            padding=1, inplace=True)

    def psp_forward(self, inputs):
        x = inputs[-1]
        psp_outs = [x]
        psp_outs.extend(self.psp_modules(x))
        psp_outs = torch.cat(psp_outs, dim=1)
        return self.bottleneck(psp_outs)

    def forward(self, inputs, hw=None):
        if self._use_scale_modules:
            scaled = []
            scaled.append(self.fpn1(inputs[0]))
            scaled.append(self.fpn2(inputs[1]))
            scaled.append(self.fpn3(inputs[2]))
            scaled.append(self.fpn4(inputs[3]))
            inputs = scaled

        laterals = [lat_conv(inputs[i])
                    for i, lat_conv in enumerate(self.lateral_convs)]
        laterals.append(self.psp_forward(inputs))

        used_backbone_levels = len(laterals)
        for i in range(used_backbone_levels - 1, 0, -1):
            prev_shape = laterals[i - 1].shape[2:]
            laterals[i - 1] = laterals[i - 1] + F.interpolate(
                laterals[i], size=prev_shape,
                mode='bilinear', align_corners=self.align_corners)

        fpn_outs = [self.fpn_convs[i](laterals[i])
                    for i in range(used_backbone_levels - 1)]
        fpn_outs.append(laterals[-1])

        for i in range(used_backbone_levels - 1, 0, -1):
            fpn_outs[i] = F.interpolate(
                fpn_outs[i], size=fpn_outs[0].shape[2:],
                mode='bilinear', align_corners=self.align_corners)

        feats = self.fpn_bottleneck(torch.cat(fpn_outs, dim=1))
        return feats


class SegmentationHead(nn.Module):
    """Classification head: Identity → Dropout → Conv2d."""
    def __init__(self, in_channels, num_classes, dropout=0.0):
        super().__init__()
        dropout_layer = nn.Identity() if dropout == 0 else nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Identity(),
            dropout_layer,
            nn.Conv2d(in_channels, num_classes, 1),
        )

    def forward(self, x):
        return self.head(x)


# ══════════════════════════════════════════════════════════════
#  Loss Functions
# ══════════════════════════════════════════════════════════════

class DiceLoss(nn.Module):
    def __init__(self, num_classes=2, ignore_index=-1, smooth=1.0):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.smooth       = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        valid = (targets != self.ignore_index)
        targets_clean = targets.clone()
        targets_clean[~valid] = 0
        one_hot = F.one_hot(targets_clean, self.num_classes).permute(0, 3, 1, 2).float()
        valid_mask = valid.unsqueeze(1).float()
        one_hot = one_hot * valid_mask
        probs   = probs * valid_mask
        dims = (0, 2, 3)
        intersection = (probs * one_hot).sum(dims)
        union = probs.sum(dims) + one_hot.sum(dims)
        dice = 1.0 - (2.0 * intersection + self.smooth) / (union + self.smooth)
        return dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.register_buffer('weight', weight)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             ignore_index=self.ignore_index, reduction='none')
        pt = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


class BoundaryLoss(nn.Module):
    def __init__(self, num_classes=2, ignore_index=-1, kernel_size=5):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.ks = kernel_size
        self.pad = kernel_size // 2

    def _get_boundary_mask(self, targets):
        t = targets.clone().float()
        t[targets == self.ignore_index] = 0
        t = t.unsqueeze(1)
        pooled = F.avg_pool2d(t, self.ks, stride=1, padding=self.pad)
        boundary = (pooled.squeeze(1) != t.squeeze(1)).float()
        valid = (targets != self.ignore_index).float()
        return boundary * valid

    def forward(self, logits, targets):
        boundary = self._get_boundary_mask(targets)
        if boundary.sum() < 1:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)
        ce = F.cross_entropy(logits, targets, ignore_index=self.ignore_index,
                             reduction='none')
        return (ce * boundary).sum() / boundary.sum().clamp(min=1)


class CombinedLoss(nn.Module):
    def __init__(self, num_classes=2, class_weights=None, ignore_index=-1,
                 ce_weight=0.2, dice_weight=0.5, boundary_weight=0.3):
        super().__init__()
        w = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.ce_weight       = ce_weight
        self.dice_weight     = dice_weight
        self.boundary_weight = boundary_weight
        self.ce       = nn.CrossEntropyLoss(weight=w, ignore_index=ignore_index)
        self.dice     = DiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.boundary = BoundaryLoss(num_classes=num_classes, ignore_index=ignore_index)

    def forward(self, logits, targets):
        return (self.ce_weight       * self.ce(logits, targets)
              + self.dice_weight     * self.dice(logits, targets)
              + self.boundary_weight * self.boundary(logits, targets))


def compute_boundary_iou(pred, target, ignore_index=-1, dilation=2):
    """Compute boundary IoU — the competition metric.

    Boundary = pixels INSIDE the mask within `dilation` pixels of the edge:
        boundary(mask) = mask & ~binary_erosion(mask, iterations=dilation)

    The previous (buggy) version used binary_dilation which produced an OUTER
    ring that is by definition outside the mask. This made gt_mask[boundary_region]
    always False, so intersection = 0 and IoU = 0 for every sample.
    """
    from scipy import ndimage
    struct = ndimage.generate_binary_structure(2, 1)
    ious = []
    ignore = (target == ignore_index)
    for cls in range(2):
        gt_mask   = (target == cls)
        pred_mask = (pred == cls)
        # Inner boundary ring: pixels in the mask removed by erosion
        gt_eroded   = ndimage.binary_erosion(gt_mask,   struct, iterations=dilation)
        pred_eroded = ndimage.binary_erosion(pred_mask, struct, iterations=dilation)
        gt_boundary   = gt_mask   & ~gt_eroded
        pred_boundary = pred_mask & ~pred_eroded
        boundary_region = (gt_boundary | pred_boundary) & ~ignore
        if boundary_region.sum() == 0:
            continue
        gt_on_boundary   = gt_mask[boundary_region]
        pred_on_boundary = pred_mask[boundary_region]
        inter = (gt_on_boundary & pred_on_boundary).sum()
        union = (gt_on_boundary | pred_on_boundary).sum()
        if union > 0:
            ious.append(inter / union)
    return float(np.mean(ious)) if ious else 0.0


class AuxFCNHead(nn.Module):
    """Lightweight FCN auxiliary head for deep supervision.

    Attached to an intermediate encoder feature (feats[2] = block 17)
    to provide a shortcut gradient path to the encoder.
    Matches MMSeg FCNHead(num_convs=1, channels=256).

    """

    def __init__(self, in_channels, num_classes, mid_channels=256, dropout=0.1):
        super().__init__()
        self.conv = ConvModule(in_channels, mid_channels, 3, padding=1, inplace=False)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.cls  = nn.Conv2d(mid_channels, num_classes, 1)

    def forward(self, x):
        return self.cls(self.drop(self.conv(x)))


print('Architecture modules defined:')
print(f'  ViTEncoder  — {TIMM_MODEL} ({EMBED_DIM}-dim, {DEPTH} layers, patch={PATCH_SIZE})')
print('  UperNetDecoder — scale_modules + PSP + FPN')
print('  SegmentationHead — dropout + Conv2d classifier')
print(f'  AuxFCNHead — on feats[2] (block {OUT_INDICES[2]}), weight={AUX_LOSS_WEIGHT}')
print('  Losses: CombinedLoss (CE=0.2 + Dice=0.5 + Boundary=0.3) + Aux CE')

In [ ]:

# ══════════════════════════════════════════════════════════════════════════
#  FloodSegModel — Lightning module with Sen1Floods11 weight loading
#
#  Downloads the FULL finetuned checkpoint from HuggingFace and loads
#  weights into encoder (ViT-Large), decoder (UperNet), and head.
#
#  TerraTorch checkpoint structure:
#    model.encoder.*  → ViT backbone
#    model.decoder.*  → UperNet decoder (scale_modules + PSP + FPN)
#    model.head.*     → SegmentationHead (Dropout + Conv2d classifier)
# ══════════════════════════════════════════════════════════════════════════

class FloodSegModel(pl.LightningModule):
    """ViT-Large encoder + UperNet decoder + SegmentationHead.

    Loads the full Prithvi-EO-2.0-300M-TL-Sen1Floods11 checkpoint,
    transferring encoder, decoder, AND head weights.
    """

    def __init__(self, in_channels, num_classes=2,
                 decoder_channels=256, head_dropout=0.1,
                 lr=3e-5, weight_decay=0.05, class_weights=None,
                 freeze_backbone=False, img_size=224,
                 warmup_iters=50, warmup_ratio=1e-6,
                 channel_names=None, transfer_mode='full'):
        super().__init__()
        self.save_hyperparameters(ignore=['channel_names'])
        self._channel_names = channel_names

        # ── Encoder: ViT-Large ──
        self.encoder = ViTEncoder(
            timm_name=TIMM_MODEL,
            in_channels=in_channels,
            img_size=img_size,
            pretrained=False,
            out_indices=OUT_INDICES,
        )

        # ── Decoder: UperNet ──
        self.decoder = UperNetDecoder(
            embed_dim=EMBED_DIM,
            pool_scales=(1, 2, 3, 6),
            channels=decoder_channels,
            align_corners=True,
            scale_modules=True,
        )

        # ── Head: Separate classification head ──
        self.seg_head = SegmentationHead(
            in_channels=decoder_channels,
            num_classes=num_classes,
            dropout=head_dropout,
        )

        # ── Auxiliary Head: on encoder feats[2] (block 17) for deep supervision ──
        self.aux_head = AuxFCNHead(
            in_channels=EMBED_DIM,
            num_classes=num_classes,
            mid_channels=256,
            dropout=head_dropout,
        )

        # ── Load Sen1Floods11 finetuned weights (if enabled) ──
        if LOAD_SEN1FLOODS11:
            self._load_sen1floods11_weights(
                in_channels=in_channels,
                channel_names=channel_names,
                transfer_mode=transfer_mode,
            )
        else:
            print('\n  LOAD_SEN1FLOODS11=False → skipping pretrained weights (random init)')

        # ── Freeze backbone if requested ──
        if freeze_backbone or transfer_mode == 'frozen':
            for p in self.encoder.parameters():
                p.requires_grad = False
            print(f'  Backbone FROZEN ({sum(1 for _ in self.encoder.parameters())} params)')

        # ── Loss & Metrics ──
        self.loss_fn    = CombinedLoss(
            num_classes=num_classes, class_weights=class_weights,
            ignore_index=-1, ce_weight=0.2, dice_weight=0.5,
            boundary_weight=0.3)  # sum=1.0
        w = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.aux_loss_fn = nn.CrossEntropyLoss(weight=w, ignore_index=-1)
        self.train_miou = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.val_miou   = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.test_miou  = MulticlassJaccardIndex(num_classes, ignore_index=-1)
        self.test_acc   = MulticlassAccuracy(num_classes, ignore_index=-1)

    def _load_sen1floods11_weights(self, in_channels, channel_names, transfer_mode):
        """Download and load weights from Prithvi-EO-2.0-300M-TL-Sen1Floods11.

        Handles:
          1. Downloading the .pt file from HuggingFace
          2. Parsing TerraTorch checkpoint format (model.encoder/decoder/head)
          3. Smart patch embed transfer for band adaptation
          4. Loading decoder + head weights (if transfer_mode='full' or 'frozen')
        """
        from huggingface_hub import hf_hub_download

        print(f'\n  Loading Sen1Floods11 finetuned weights...')
        print(f'  Source: {HF_REPO}/{HF_CKPT_FILE}')
        print(f'  Transfer mode: {transfer_mode}')

        try:
            path = hf_hub_download(repo_id=HF_REPO, filename=HF_CKPT_FILE)
            ckpt = torch.load(path, map_location='cpu', weights_only=False)
        except Exception as e:
            print(f'  ⚠ Download failed: {e}')
            print(f'  Using random initialization instead.')
            return

        # ── Parse checkpoint format ──
        if 'state_dict' in ckpt:
            raw_sd = ckpt['state_dict']
        elif 'model' in ckpt:
            raw_sd = ckpt['model']
        else:
            raw_sd = ckpt

        prefixes = set()
        for k in list(raw_sd.keys())[:50]:
            prefixes.add(k.split('.')[0])
        print(f'  Checkpoint key prefixes: {sorted(prefixes)}')
        print(f'  Total keys in checkpoint: {len(raw_sd)}')

        # ── Classify keys into encoder / decoder / head ──
        encoder_sd = {}
        decoder_sd = {}
        head_sd    = {}

        for k, v in raw_sd.items():
            # TerraTorch format: model.encoder.*, model.decoder.*, model.head.*
            if k.startswith('model.encoder.'):
                encoder_sd[k.replace('model.encoder.', '')] = v
            elif k.startswith('model.decoder.'):
                decoder_sd[k.replace('model.decoder.', '')] = v
            elif k.startswith('model.head.'):
                head_sd[k.replace('model.head.', '')] = v
            # mmseg format: backbone.*, decode_head.*
            elif k.startswith('backbone.'):
                encoder_sd[k.replace('backbone.', '')] = v
            elif k.startswith('decode_head.'):
                decoder_sd[k.replace('decode_head.', '')] = v
            # Flat format (no prefix)
            elif not any(k.startswith(p) for p in ('optimizer', 'scheduler', 'epoch',
                                                     'global_step', 'callbacks', 'loops',
                                                     'lr_schedulers', 'hparams')):
                if any(kw in k for kw in ('psp_modules', 'lateral_convs', 'fpn_convs',
                                           'fpn_bottleneck', 'bottleneck',
                                           'fpn1', 'fpn2', 'fpn3', 'fpn4')):
                    decoder_sd[k] = v
                elif any(kw in k for kw in ('seg_head', 'classifier')):
                    head_sd[k] = v
                else:
                    encoder_sd[k] = v
                                    
        print(f'  Encoder keys: {len(encoder_sd)} | Decoder keys: {len(decoder_sd)} | Head keys: {len(head_sd)}')

        # ── Load encoder weights ──
        self._load_encoder_weights(encoder_sd, in_channels, channel_names)

        # ── Load decoder + head weights ──
        if transfer_mode in ('full', 'frozen'):
            if decoder_sd:
                self._load_decoder_weights(decoder_sd)
            else:
                print(f'  Decoder: no pretrained keys found — using random init')
            if head_sd:
                self._load_head_weights(head_sd)
            else:
                print(f'  Head: no pretrained keys found — using random init')
        elif transfer_mode == 'encoder':
            print(f'  Decoder + Head: random init (transfer_mode="encoder")')

    def _load_encoder_weights(self, pretrained_sd, in_channels, channel_names):
        """Load encoder weights with smart patch embed transfer."""
        cur = self.encoder.vit.state_dict()
        mapped = {}

        for k, v in pretrained_sd.items():
            nk = k
            for pfx in ('encoder.', 'vit.', 'model.'):
                if nk.startswith(pfx):
                    nk = nk[len(pfx):]

            if nk not in cur:
                continue

            if v.shape == cur[nk].shape:
                mapped[nk] = v
            elif 'patch_embed' in nk and 'weight' in nk:
                pt_w = v
                print(f'  Patch embed shapes — pretrained: {pt_w.shape}, ours: {cur[nk].shape}')

                if pt_w.ndim == 5:
                    pt_w = pt_w.squeeze(2)
                    print(f'  Patch embed: squeezed 5D→4D: {pt_w.shape}')

                pretrained_chans = pt_w.shape[1]
                needed_chans = cur[nk].shape[1]

                if channel_names is not None:
                    new_weight = cur[nk].clone()
                    matched = []
                    for our_idx, band_name in enumerate(channel_names):
                        pt_idx = BAND_TO_PRETRAINED_INDEX.get(band_name)
                        if pt_idx is not None and pt_idx < pretrained_chans:
                            new_weight[:, our_idx, :, :] = pt_w[:, pt_idx, :, :]
                            matched.append(f'{band_name}(ours[{our_idx}]←PT[{pt_idx}])')
                    mapped[nk] = new_weight
                    unmatched = [b for b in channel_names if b not in BAND_TO_PRETRAINED_INDEX]
                    print(f'  Patch embed: SMART TRANSFER — {len(matched)}/{needed_chans} bands matched')
                    for m in matched:
                        print(f'    ✓ {m}')
                    if unmatched:
                        print(f'    ✗ Random init: {unmatched}')
                else:
                    if pretrained_chans == needed_chans:
                        mapped[nk] = pt_w
                    else:
                        print(f'  Patch embed: SKIPPED ({pretrained_chans}→{needed_chans}ch, no channel_names)')
            elif nk == 'pos_embed':
                old_pe = v
                cls_tok = old_pe[:, :1, :]
                pos_tok = old_pe[:, 1:, :]
                old_gs = int(pos_tok.shape[1] ** 0.5)
                new_gs = int((cur[nk].shape[1] - 1) ** 0.5)
                if old_gs != new_gs:
                    pos_tok = pos_tok.reshape(1, old_gs, old_gs, -1).permute(0, 3, 1, 2)
                    pos_tok = F.interpolate(pos_tok, size=(new_gs, new_gs),
                                            mode='bicubic', align_corners=False)
                    pos_tok = pos_tok.permute(0, 2, 3, 1).reshape(1, new_gs * new_gs, -1)
                    mapped[nk] = torch.cat([cls_tok, pos_tok], dim=1)
                    print(f'  Pos embed: interpolated {old_gs}→{new_gs}')
                else:
                    mapped[nk] = v

        info = self.encoder.vit.load_state_dict(mapped, strict=False)
        print(f'  ✓ Encoder: {len(mapped)}/{len(cur)} params loaded')
        missing = [k for k in info.missing_keys
                   if not any(kw in k for kw in ('pos_embed', 'patch_embed'))]
        if missing:
            print(f'    Missing: {missing[:8]}...' if len(missing) > 8 else f'    Missing: {missing}')

    def _load_decoder_weights(self, pretrained_sd):
        """Load UperNet decoder weights from the finetuned checkpoint."""
        cur = self.decoder.state_dict()
        mapped = {}

        for k, v in pretrained_sd.items():
            nk = k
            for pfx in ('decoder.', 'decode_head.'):
                if nk.startswith(pfx):
                    nk = nk[len(pfx):]
            if nk in cur and v.shape == cur[nk].shape:
                mapped[nk] = v

        if mapped:
            info = self.decoder.load_state_dict(mapped, strict=False)
            print(f'  ✓ Decoder: {len(mapped)}/{len(cur)} params loaded (pretrained UperNet)')
            missing = info.missing_keys
            if missing:
                print(f'    Missing: {missing[:10]}...' if len(missing) > 10
                      else f'    Missing: {missing}')
        else:
            print(f'  ⚠ Decoder: no key matches found — using random init')

    def _load_head_weights(self, pretrained_sd):
        """Load SegmentationHead weights from the finetuned checkpoint."""
        cur = self.seg_head.state_dict()
        mapped = {}

        for k, v in pretrained_sd.items():
            nk = k
            for pfx in ('head.', 'seg_head.'):
                if nk.startswith(pfx):
                    nk = nk[len(pfx):]
            if nk in cur and v.shape == cur[nk].shape:
                mapped[nk] = v

        if mapped:
            info = self.seg_head.load_state_dict(mapped, strict=False)
            print(f'  ✓ Head: {len(mapped)}/{len(cur)} params loaded (pretrained classifier)')
            if info.missing_keys:
                print(f'    Missing: {info.missing_keys}')
        else:
            print(f'  Head: no key matches — using random init')

    # ── Forward ──
    def forward(self, x):
        feats   = self.encoder(x)
        decoded = self.decoder(feats)
        logits  = self.seg_head(decoded)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, x.shape[2:],
                                   mode='bilinear', align_corners=True)
        return logits

    # ── Training step (on RandomCrop patches) ──
    def _step(self, batch, stage):
        x = batch['image']
        feats   = self.encoder(x)
        decoded = self.decoder(feats)
        logits  = self.seg_head(decoded)
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, x.shape[2:],
                                   mode='bilinear', align_corners=True)

        main_loss = self.loss_fn(logits, batch['mask'])

        # Auxiliary loss on encoder feats[2] (block 17)
        aux_out = self.aux_head(feats[2])
        if aux_out.shape[-2:] != x.shape[-2:]:
            aux_out = F.interpolate(aux_out, x.shape[2:],
                                    mode='bilinear', align_corners=True)
        aux_loss = self.aux_loss_fn(aux_out, batch['mask'])

        loss = main_loss + AUX_LOSS_WEIGHT * aux_loss

        preds = logits.argmax(1)
        if stage == 'train':
            self.train_miou(preds, batch['mask'])
            self.log('train/loss', loss, prog_bar=True)
            self.log('train/main_loss', main_loss, on_step=False, on_epoch=True)
            self.log('train/aux_loss', aux_loss, on_step=False, on_epoch=True)
            self.log('train/mIoU', self.train_miou,
                     on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # ── Sliding-window val/test step (matches submission exactly) ──
    def _sw_step(self, batch, stage):
        """Sliding-window inference on full 512×512 images — same as submission."""
        imgs  = batch['image']   # (B, C, H, W) — full resolution
        masks = batch['mask']    # (B, H, W)
        B, C, H, W = imgs.shape
        ws     = self.hparams.img_size   # 224 — model's native window
        stride = ws // 2                 # 112

        all_preds  = []
        all_logits = []
        for b in range(B):
            img       = imgs[b]   # (C, H, W)
            logit_sum = torch.zeros(2, H, W, device=img.device)
            cnt       = torch.zeros(1, H, W, device=img.device)

            # Build grid of top-left corners ensuring full coverage
            ys = list(range(0, H - ws + 1, stride))
            if not ys or ys[-1] + ws < H:
                ys.append(H - ws)
            xs = list(range(0, W - ws + 1, stride))
            if not xs or xs[-1] + ws < W:
                xs.append(W - ws)

            for y in ys:
                for x in xs:
                    patch = img[:, y:y+ws, x:x+ws].unsqueeze(0)
                    out   = self(patch).squeeze(0)  # (2, ws, ws) — logits
                    logit_sum[:, y:y+ws, x:x+ws] += out
                    cnt[:,       y:y+ws, x:x+ws] += 1

            avg_logits = logit_sum / cnt.clamp(min=1)  # (2, H, W)
            pred = avg_logits.argmax(0)                 # (H, W)
            all_preds.append(pred)
            all_logits.append(avg_logits)

        preds      = torch.stack(all_preds)   # (B, H, W)
        logits_all = torch.stack(all_logits)  # (B, 2, H, W)

        # Compute loss on sliding-window averaged logits
        loss = self.loss_fn(logits_all, masks)

        if stage == 'val':
            self.val_miou(preds, masks)
            self.log('val/loss', loss,
                     on_step=False, on_epoch=True, prog_bar=True)
            self.log('val/mIoU', self.val_miou,
                     on_step=False, on_epoch=True, prog_bar=True)
            pred_np = preds.cpu().numpy()
            mask_np = masks.cpu().numpy()
            b_ious  = [compute_boundary_iou(pred_np[i], mask_np[i]) for i in range(B)]
            self.log('val/boundary_mIoU', sum(b_ious) / max(len(b_ious), 1),
                     on_step=False, on_epoch=True, prog_bar=True)
        else:  # test
            self.test_miou(preds, masks)
            self.test_acc(preds, masks)
            self.log('test/loss', loss, on_step=False, on_epoch=True)
            self.log('test/mIoU', self.test_miou, on_step=False, on_epoch=True)
            self.log('test/Overall_Accuracy', self.test_acc, on_step=False, on_epoch=True)
            pred_np = preds.cpu().numpy()
            mask_np = masks.cpu().numpy()
            b_ious  = [compute_boundary_iou(pred_np[i], mask_np[i]) for i in range(B)]
            self.log('test/boundary_mIoU', sum(b_ious) / max(len(b_ious), 1),
                     on_step=False, on_epoch=True)

    def training_step(self, batch, batch_idx):   return self._step(batch, 'train')
    def validation_step(self, batch, batch_idx): self._sw_step(batch, 'val')
    def test_step(self, batch, batch_idx):       self._sw_step(batch, 'test')

    def configure_optimizers(self):
        """AdamW + Cosine Annealing with linear warmup.

        Layer-wise LR decay for 24-layer ViT-Large:
          - Deeper encoder layers → lower LR (knowledge preservation)
          - Patch embed (partially random) → 10× base LR
          - Decoder + Head → 10× base LR (training from scratch)
        """
        lr_decay = 0.75
        base_lr  = self.hparams.lr
        param_groups = []
        no_decay_keywords = ['bias', 'norm', 'bn', 'scale']
        num_layers = DEPTH   # 24

        for name, param in self.encoder.named_parameters():
            if not param.requires_grad:
                continue
            layer_id = num_layers
            if 'patch_embed' in name:
                layer_id = num_layers + 1
            elif 'pos_embed' in name:
                layer_id = 0
            elif 'blocks.' in name:
                try:
                    layer_id = int(name.split('blocks.')[1].split('.')[0])
                except (IndexError, ValueError):
                    layer_id = num_layers

            scale = lr_decay ** (num_layers - layer_id)
            if 'patch_embed' in name:
                scale = 10.0

            is_decay = not any(kw in name for kw in no_decay_keywords)
            wd = self.hparams.weight_decay if is_decay else 0.0
            param_groups.append({
                'params': [param], 'lr': base_lr * scale,
                'weight_decay': wd, 'name': f'enc.{name}',
            })

        # Decoder + Head at 5× base LR (pretrained from Sen1Floods11)
        for name, param in self.decoder.named_parameters():
            if not param.requires_grad:
                continue
            is_decay = not any(kw in name for kw in no_decay_keywords)
            wd = self.hparams.weight_decay if is_decay else 0.0
            param_groups.append({
                'params': [param], 'lr': base_lr * 5.0,
                'weight_decay': wd, 'name': f'dec.{name}',
            })

        for name, param in self.seg_head.named_parameters():
            if not param.requires_grad:
                continue
            is_decay = not any(kw in name for kw in no_decay_keywords)
            wd = self.hparams.weight_decay if is_decay else 0.0
            param_groups.append({
                'params': [param], 'lr': base_lr * 5.0,
                'weight_decay': wd, 'name': f'head.{name}',
            })

        # Auxiliary head at 5× base LR (same as decoder/head)
        for name, param in self.aux_head.named_parameters():
            if not param.requires_grad:
                continue
            is_decay = not any(kw in name for kw in no_decay_keywords)
            wd = self.hparams.weight_decay if is_decay else 0.0
            param_groups.append({
                'params': [param], 'lr': base_lr * 5.0,
                'weight_decay': wd, 'name': f'aux.{name}',
            })

        opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))

        total_steps  = self.trainer.estimated_stepping_batches
        warmup_iters = self.hparams.get('warmup_iters', 50)

        def lr_lambda(step):
            if step < warmup_iters:
                return max(self.hparams.get('warmup_ratio', 1e-6),
                           step / max(1, warmup_iters))
            progress = (step - warmup_iters) / max(1, total_steps - warmup_iters)
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            opt, [lr_lambda] * len(param_groups))

        return {
            'optimizer': opt,
            'lr_scheduler': {'scheduler': scheduler, 'interval': 'step', 'frequency': 1},
        }


# ── Factory ──
def create_model():
    '''Create FloodSegModel for the 300M-TL experiment.'''
    model = FloodSegModel(
        in_channels      = N_CH,
        num_classes       = NUM_CLASSES,
        decoder_channels  = DECODER_CH,
        head_dropout      = HEAD_DROPOUT,
        lr                = LR,
        weight_decay      = WEIGHT_DECAY,
        class_weights     = CLASS_WEIGHTS,
        freeze_backbone   = FREEZE_BACKBONE,
        img_size          = IMG_SIZE,
        channel_names     = ch_names,
        transfer_mode     = TRANSFER_MODE,
    )

    matched = [b for b in ch_names if b in BAND_TO_PRETRAINED_INDEX]
    fresh   = [b for b in ch_names if b not in BAND_TO_PRETRAINED_INDEX]

    print(f'\n  Architecture: ViT-Large + UperNet + SegmentationHead + AuxFCNHead')
    print(f'  Aux head: feats[2] (block {OUT_INDICES[2]}), weight={AUX_LOSS_WEIGHT}')
    print(f'  Transfer mode: {TRANSFER_MODE} | Load Sen1Floods11: {LOAD_SEN1FLOODS11}')
    print(f'  Channels ({N_CH}): {ch_names}')
    print(f'    Pretrained match: {matched}  |  Random init: {fresh}')

    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params — total: {total_p:,}  trainable: {train_p:,}')

    return model

print(f'  Source:   {HF_REPO}')
print(f'  Backbone: {TIMM_MODEL} ({EMBED_DIM}-dim, {DEPTH} layers)')
print(f'  Decoder:  UperNet (pretrained={LOAD_SEN1FLOODS11}, scale_modules + PSP + FPN)')
print(f'  Head:     SegmentationHead (pretrained={LOAD_SEN1FLOODS11})')
print(f'  AuxHead:  AuxFCNHead on feats[2] (block {OUT_INDICES[2]}), weight={AUX_LOSS_WEIGHT}')
print(f'  Transfer mode: {TRANSFER_MODE} | Load Sen1Floods11: {LOAD_SEN1FLOODS11}')
print(f'FloodSegModel defined with Sen1Floods11 weight loading (enabled={LOAD_SEN1FLOODS11}).')

---
## Training & Evaluation Engine

`run_experiment()` handles:
1. Compute per-channel normalization stats
2. Build DataModule with the channel stack
3. Create model and load backbone weights
4. Train with early stopping on `val/mIoU`
5. Test on held-out split
6. Save config, metrics, and best checkpoint

In [ ]:
def run_experiment():
    """Run train → validate → test for the 300M-TL experiment."""
    print(f'\n{"=" * 70}')
    print(f'  {EXP_CFG["desc"]}')
    print(f'  Channels ({N_CH}): {ch_names}')
    print(f'  LR: {LR} | Freeze: {FREEZE_BACKBONE}')
    print(f'{"=" * 70}\n')

    exp_dir = OUTPUT_DIR / EXP_CFG['name']
    exp_dir.mkdir(parents=True, exist_ok=True)
    pl.seed_everything(SEED)

    # ── 0. Precompute channel cache (Refined Lee + dB → .npy) ──
    all_files = list(set(train_files + val_files + test_files))
    precompute_channel_cache(all_files, IMG_DIR, EXP_CFG, label='train+val+test')
    # Cache pred files separately (different source directory)
    if pred_files:
        try:
            precompute_channel_cache(pred_files, PRED_IMG_DIR, EXP_CFG, label='pred')
        except Exception as e:
            print(f'  Warning: could not cache pred files: {e}')

    # ── 1. Normalization stats ──
    stats_path = exp_dir / 'norm_stats.json'
    if stats_path.exists():
        with open(stats_path) as f:
            cached = json.load(f)
        means, stds = cached['means'], cached['stds']
        print('  Loaded cached normalization stats.')
    else:
        means, stds = compute_norm_stats(EXP_CFG, train_files, IMG_DIR)
        with open(stats_path, 'w') as f:
            json.dump({'means': means, 'stds': stds, 'channels': ch_names}, f, indent=2)
    for i, name in enumerate(ch_names):
        print(f'    {name:>6s}  mean={means[i]:>10.4f}  std={stds[i]:>10.4f}')

    # ── 2. DataModule ──
    dm = FloodDataModule(
        exp_cfg=EXP_CFG, means=means, stds=stds,
        train_files=train_files, val_files=val_files,
        test_files=test_files, pred_files=pred_files,
        img_dir=IMG_DIR, lbl_dir=LBL_DIR,
        pred_img_dir=PRED_IMG_DIR,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        img_size=IMG_SIZE,
    )
    dm.setup('fit')
    sample = dm.train_ds[0]
    print(f'\n  Train samples: {len(dm.train_ds)} | Val: {len(dm.val_ds)}')
    print(f'  Sample image : {sample["image"].shape}  dtype={sample["image"].dtype}')
    print(f'  Sample mask  : {sample["mask"].shape}   unique={sample["mask"].unique().tolist()}')

    # ── 3. Model ──
    model = create_model()

    # ── 4. Callbacks & Trainer ──
    ckpt_cb = ModelCheckpoint(
        monitor='val/mIoU', mode='max',
        dirpath=str(exp_dir / 'checkpoints'),
        filename='best-epoch{epoch:02d}',
        save_top_k=1, save_last=False,  # only keep the single best checkpoint
    )
    lr_cb   = LearningRateMonitor(logging_interval='step')
    es_cb   = EarlyStopping(monitor='val/mIoU', mode='max', patience=ES_PATIENCE, verbose=True)
    logger  = CSVLogger(save_dir=str(exp_dir), name='logs')

    # ── TPU vs GPU/CPU trainer config ──
    if USE_TPU and _XLA_AVAILABLE:
        trainer = pl.Trainer(
            accelerator='tpu', devices=TPU_CORES,
            precision='bf16-true',
            max_epochs=EPOCHS,
            accumulate_grad_batches=ACCUM_GRAD,
            check_val_every_n_epoch=1,
            log_every_n_steps=5,
            callbacks=[ckpt_cb, lr_cb, es_cb],
            logger=logger,
            enable_checkpointing=True,
            gradient_clip_val=1.0,
        )
    else:
        trainer = pl.Trainer(
            accelerator='auto', strategy='auto', devices='auto',
            precision='16-mixed',
            max_epochs=EPOCHS,
            accumulate_grad_batches=ACCUM_GRAD,
            check_val_every_n_epoch=1,
            log_every_n_steps=5,
            callbacks=[ckpt_cb, lr_cb, es_cb],
            logger=logger,
            enable_checkpointing=True,
            gradient_clip_val=1.0,
        )

    # ── 5. Train ──
    t0 = time.time()
    trainer.fit(model, datamodule=dm)
    train_min = (time.time() - t0) / 60
    print(f'\n  Training done in {train_min:.1f} min')
    print(f'  Best val/mIoU = {ckpt_cb.best_model_score:.4f}')
    print(f'  Checkpoint    = {ckpt_cb.best_model_path}')

    # ── 6. Test ──
    dm.setup('test')
    test_out = trainer.test(model, datamodule=dm, ckpt_path=ckpt_cb.best_model_path)

    # ── 7. Save results ──
    results = {
        'experiment': EXP_CFG['name'],
        'desc': EXP_CFG['desc'],
        'model': HF_REPO,
        'backbone': BACKBONE,
        'timm_model': TIMM_MODEL,
        'embed_dim': EMBED_DIM,
        'depth': DEPTH,
        'patch_size': PATCH_SIZE,
        'decoder': DECODER,
        'freeze_backbone': FREEZE_BACKBONE,
        'img_size': IMG_SIZE,
        'channels': ch_names,
        'n_channels': N_CH,
        'lr': LR,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'accum_grad': ACCUM_GRAD,
        'best_val_miou': float(ckpt_cb.best_model_score) if ckpt_cb.best_model_score else 0,
        'test_metrics': test_out[0] if test_out else {},
        'train_time_min': round(train_min, 2),
        'best_ckpt': ckpt_cb.best_model_path,
        'epochs_run': trainer.current_epoch,
    }

    with open(exp_dir / 'results.json', 'w') as f:
        json.dump(results, f, indent=2)

    return results, trainer, model, dm, ckpt_cb.best_model_path

print('run_experiment() defined.')


In [ ]:
# ══════════════════════════════════════════════════════════════
#  EXECUTE — Train the 300M-TL model
# ══════════════════════════════════════════════════════════════

results, trainer, model, dm, best_ckpt = run_experiment()

print('\n✓ Training complete.')

---
## Visualize Predictions on Test Set

In [ ]:

# ── Visualize predictions vs ground truth ──
ckpt_state = torch.load(best_ckpt, map_location='cpu')
model.load_state_dict(ckpt_state['state_dict'])
model.eval()
device = get_device()
model = model.to(device)

dm.setup('test')
n_show = min(6, len(dm.test_ds))

fig, axes = plt.subplots(n_show, 3, figsize=(14, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

import matplotlib.colors as mcolors
flood_cmap = mcolors.ListedColormap(['#2c3e50', '#e74c3c'])

ws     = IMG_SIZE      # 224 — patch_size=14, must be divisible (224/14=16 ✓, 512/14 ✗)
stride = ws // 2       # 112

def _sliding_pred(img_t, device):
    """Sliding-window prediction on a (1, C, H, W) full-resolution tensor."""
    img = img_t.squeeze(0)   # (C, H, W)
    C, H, W = img.shape
    logit_sum = torch.zeros(2, H, W, device='cpu')
    cnt       = torch.zeros(1, H, W, device='cpu')
    ys = list(range(0, H - ws + 1, stride));
    if not ys or ys[-1] + ws < H: ys.append(H - ws)
    xs = list(range(0, W - ws + 1, stride));
    if not xs or xs[-1] + ws < W: xs.append(W - ws)
    for y in ys:
        for x in xs:
            patch = img[:, y:y+ws, x:x+ws].unsqueeze(0).to(device)
            out   = model(patch).squeeze(0).cpu()
            logit_sum[:, y:y+ws, x:x+ws] += out
            cnt[:,       y:y+ws, x:x+ws] += 1
    return (logit_sum / cnt.clamp(min=1)).argmax(0).numpy()  # (H, W)

with torch.no_grad():
    for i in range(n_show):
        sample = dm.test_ds[i]
        img_t  = sample['image'].unsqueeze(0).to(device)
        mask   = sample['mask'].numpy()
        pred   = _sliding_pred(img_t, device)

        H, W = pred.shape  # 512×512

        band0 = sample['image'][0].numpy()
        axes[i, 0].imshow(band0, cmap='gray')
        axes[i, 0].set_ylabel(sample['filename'], fontsize=8)
        if i == 0: axes[i, 0].set_title('Input (band 0)', fontweight='bold')

        axes[i, 1].imshow(mask, cmap=flood_cmap, vmin=0, vmax=1)
        if i == 0: axes[i, 1].set_title('Ground Truth', fontweight='bold')

        axes[i, 2].imshow(pred, cmap=flood_cmap, vmin=0, vmax=1)
        iou_1 = np.logical_and(pred == 1, mask == 1).sum() / (np.logical_or(pred == 1, mask == 1).sum() + 1e-8)
        title = 'Prediction' if i == 0 else f'IoU={iou_1:.3f}'
        axes[i, 2].set_title(title, fontweight='bold')
        axes[i, 2].text(W // 2, H - 10, f'IoU={iou_1:.3f}', ha='center', fontsize=9,
                        color='white', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        for c in range(3):
            axes[i, c].set_xticks([]); axes[i, c].set_yticks([])

fig.suptitle(f'Prithvi-EO-2.0-300M-TL-Sen1Floods11 | {EXP_CFG["desc"]}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / EXP_CFG['name'] / 'val_predictions.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Prediction & Kaggle Submission

Uses sliding window inference for full 512×512 coverage when training at 224×224.

**CRITICAL:** Uses column-major (`order='F'`) RLE encoding per the competition helper notebook.

In [ ]:
def mask_to_rle(mask):
    '''Convert binary 2D mask to RLE string (column-major / Fortran order, 1-indexed).'''
    pixels = mask.flatten(order='F')  # ← COLUMN-MAJOR (matches helper notebook)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def sliding_window_inference(model_obj, image_tensor, window_size, stride, device, num_classes=2):
    """Sliding window inference on (C, H, W) tensor → (H, W) prediction."""
    C, H, W = image_tensor.shape
    count  = torch.zeros(num_classes, H, W, device='cpu')
    logits = torch.zeros(num_classes, H, W, device='cpu')

    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y_end = min(y + window_size, H)
            x_end = min(x + window_size, W)
            y_start = max(0, y_end - window_size)
            x_start = max(0, x_end - window_size)

            crop = image_tensor[:, y_start:y_end, x_start:x_end].unsqueeze(0).to(device)
            with torch.no_grad():
                out = model_obj(crop).squeeze(0).cpu()
            logits[:, y_start:y_end, x_start:x_end] += out
            count[:, y_start:y_end, x_start:x_end]  += 1

    logits /= count.clamp(min=1)
    return logits.argmax(0).numpy()


def generate_submission(model_obj, data_module, ckpt_path, out_dir):
    '''Generate predictions and create submission CSV.'''
    print('\nGenerating predictions for submission...')

    ckpt_state = torch.load(ckpt_path, map_location='cpu')
    model_obj.load_state_dict(ckpt_state['state_dict'])
    model_obj.eval()
    device = get_device()
    model_obj = model_obj.to(device)

    use_sliding = IMG_SIZE < 512
    if use_sliding:
        stride = IMG_SIZE // 2
        print(f'  Sliding window: window={IMG_SIZE}, stride={stride}')

    if use_sliding:
        all_preds, all_fnames = [], []
        exp_cfg = data_module.exp_cfg
        means = np.array(data_module.means, dtype=np.float32).reshape(-1, 1, 1)
        stds  = np.array(data_module.stds,  dtype=np.float32).reshape(-1, 1, 1)

        cache_dir = _get_cache_dir()
        for fname in pred_files:
            channels = load_cached_channels(fname, cache_dir)
            if channels is None:
                with rasterio.open(PRED_IMG_DIR / f'{fname}_image.tif') as src:
                    img6 = src.read().astype(np.float32)
                channels = build_channel_stack(img6, None, exp_cfg)
            channels = (channels - means) / (stds + 1e-8)
            channels = torch.from_numpy(channels).float()
            pred = sliding_window_inference(model_obj, channels, IMG_SIZE, stride, device)
            all_preds.append(pred)
            all_fnames.append(fname)
    else:
        data_module.setup('predict')
        loader = data_module.predict_dataloader()
        all_preds, all_fnames = [], []
        with torch.no_grad():
            for batch in loader:
                images = batch['image'].to(device)
                fnames = batch['filename']
                logits = model_obj(images)
                preds = logits.argmax(dim=1).cpu().numpy()
                for i in range(preds.shape[0]):
                    all_preds.append(preds[i])
                    all_fnames.append(fnames[i])

    rows = []
    for pred_mask, fname in zip(all_preds, all_fnames):
        binary = (pred_mask > 0).astype(np.uint8)
        rle = mask_to_rle(binary)
        sample_id = fname.replace('_image', '')
        rows.append({'id': sample_id, 'rle_mask': rle})

    df = pd.DataFrame(rows)
    df['rle_mask'] = df['rle_mask'].replace('', ' ')
    csv_path = out_dir / f'submission_{EXP_CFG["name"]}.csv'
    df.to_csv(csv_path, index=False)
    print(f'  Submission saved: {csv_path} ({len(rows)} patches)')

    # Save prediction TIFs
    tif_dir = out_dir / 'pred_tifs'
    tif_dir.mkdir(exist_ok=True)
    for pred_mask, fname in zip(all_preds, all_fnames):
        ref_path = PRED_IMG_DIR / f'{fname}_image.tif'
        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()
        meta.update({'count': 1, 'dtype': 'int16', 'nodata': -1, 'compress': 'lzw'})
        out_path = tif_dir / f'{fname}_pred.tif'
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(pred_mask.astype(np.int16), 1)
    print(f'  Prediction TIFs saved to: {tif_dir}')
    return csv_path

print('generate_submission() defined.')

In [ ]:
# ── Generate submission ──
sub_csv = generate_submission(
    model, dm, best_ckpt,
    OUTPUT_DIR / EXP_CFG['name']
)

print('\n✓ Submission generation complete.')

In [ ]:
# ── Copy submission to output root ──
import shutil

src = OUTPUT_DIR / EXP_CFG['name'] / f'submission_{EXP_CFG["name"]}.csv'

if ON_KAGGLE:
    dst = Path('/kaggle/working/submission.csv')
else:
    dst = OUTPUT_DIR / 'submission.csv'

if src.exists():
    shutil.copy2(src, dst)
    print(f'✓ submission.csv ready at {dst}')
else:
    print(f'⚠ {src} not found — run the experiment first.')

In [ ]:
# ── Print final results ──
rpath = OUTPUT_DIR / EXP_CFG['name'] / 'results.json'
if rpath.exists():
    with open(rpath) as f:
        r = json.load(f)
    tm = r.get('test_metrics', {})
    print(f'\n{"═" * 60}')
    print(f'  Prithvi-EO-2.0-300M-TL-Sen1Floods11 Results')
    print(f'{"═" * 60}')
    print(f'  Model:        {r["model"]}')
    print(f'  Architecture: {r["timm_model"]} (embed={r["embed_dim"]}, depth={r["depth"]})')
    print(f'  Channels:     {r["channels"]}')
    print(f'  Best Val mIoU:  {r.get("best_val_miou", 0):.4f}')
    print(f'  Test mIoU:      {tm.get("test/mIoU", 0):.4f}')
    print(f'  Test Accuracy:  {tm.get("test/Overall_Accuracy", 0):.4f}')
    print(f'  Test Boundary:  {tm.get("test/boundary_mIoU", 0):.4f}')
    print(f'  Training time:  {r.get("train_time_min", 0):.1f} min')
    print(f'  Epochs run:     {r.get("epochs_run", 0)}')
    print(f'{"═" * 60}')
else:
    print('No results found — run the experiment first.')

In [ ]:
rm -rf "/kaggle/working/experiments/channel_cache"

---
## Summary

This notebook uses the **fully finetuned** Prithvi-EO-2.0-300M-TL-Sen1Floods11 model — a ViT-Large encoder + UperNet decoder pretrained on HLS V2 satellite data and finetuned on the Sen1Floods11 flood detection benchmark.

**Key characteristics:**

1. **ViT-Large backbone** — 1024-dim, 24 layers, patch_size=16
2. **Full transfer** — encoder, decoder, AND head are all pretrained on Sen1Floods11 flood detection
3. **Moderate decoder LR (5×)** — decoder is pretrained (not random), so 5× base LR is sufficient
4. **Smart patch embed transfer** — copies pretrained filters for matching bands while randomly initializing new channels
5. **Column-major RLE** — uses `order='F'` for correct submission format
6. **Raw 6 bands** — uses HH, HV, Green, Red, NIR, SWIR directly

**Weight loading from TerraTorch `.pt` format:**
- Parses `model.encoder.*`, `model.decoder.*`, `model.head.*` keys from the Sen1Floods11 checkpoint
- Smart band-wise patch embed transfer for channel adaptation
- Loads full UperNet decoder weights (PSP + FPN + scale modules)
- Loads SegmentationHead classifier weights
- Optional `TRANSFER_MODE` control: 'full' (all), 'encoder' (backbone-only), 'frozen' (encoder frozen)
- `LOAD_SEN1FLOODS11` flag to toggle pretrained weight loading on/off

**Compared to the 600M notebook:**
- ~0.5× encoder parameters (more efficient)
- Larger batch size (4 vs 2) with lower gradient accumulation (4 vs 8)

- Pretrained flood decoder → faster convergence expected- Lower decoder/head LR (5× vs 10×) since decoder is already trained